# Herramientas instalacion


In [10]:
pip install requests
#request sirve para generar la peticion a la pagina web

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install lxml

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 927.6 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/4.1 MB 1.5 MB/s eta 0:00:03
   ------------ --------------------------- 1.3/4.1 MB 1.8 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.1 MB 1.5 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.1 MB 1.5 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.1 MB 1.2 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.1 MB 1.2 MB/s eta 0:00:02
   -------------------- ------------------- 2.1/4.1 MB 1.1 MB/s eta 0:00:02
   -------------------- ------------------- 2.1/4.1 MB 1.1 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.1 MB 999.6 kB/s eta 0:00:02
   ------------------------- 

In [7]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install lxml

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np  #Para que cuando ocupemos la libreria no usemos mas que np
import time
import random

BASE_URL = "https://books.toscrape.com/"
print("Librerias cargadas correctamente")

Librerias cargadas correctamente


# Descargar el HTML de una sola pagina
El primer paso de todo scraping es hacer una peticion HTTP

In [19]:
response = requests.get(BASE_URL, timeout = 10)
print("Codigo de estado: ", response.status_code)
print("longitud del html: ", len(response.text))

#vistaszo rapido a los primeros 500 caracteres del HTML
print (response.text[:500])

Codigo de estado:  200
longitud del html:  51294
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


# parsear el html con beautifulsoup

In [21]:
soup = BeautifulSoup(response.text, "html.parser")

#Buscamos todos los elementos <article class = "product_pod">
libros_html = soup.select("article.product_pod")
print(f"se encontraron {len(libros_html)} libro en esta pagina")

#Vemos el primero
print(libros_html[0].prettify()[:600])

se encontraron 20 libro en esta pagina
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_pric


# Extraer los datos de cada libro (titulo, precio, disponibilidad)

In [30]:
def extraer_rating(clases_css):
    """El reating viene como una clase CSS: 'star-rating Three' -> 3"""
    mapa = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for clase in clases_css:
        
        if clase in mapa:
            return mapa[clase]
    return None

libros = []
for item in libros_html:
    titulo = item.h3.a["title"]
    precio_texto = item.select_one("p.price_color").text.strip()
    precio = float(precio_texto.replace("£", "").replace("Â", ""))
    disponibilidad = item.select_one("p.instock.availability").text.strip()
    rating_clases = item.select_one("p.star-rating")["class"] # ej. ['star-rating', 'Three']
    rating = extraer_rating(rating_clases)
    enlace_relativo = item.h3.a["href"]

    libros.append({
        "titulo": titulo,
        "precio_gbp": precio,
        "disponibilidad": disponibilidad,
        "rating": rating,
        "enlace": BASE_URL + enlace_relativo
    })


df_pagina1 = pd.DataFrame(libros)
df_pagina1.head(20)
    

,titulo,precio_gbp,disponibilidad,rating,enlace
0,A Light in the Attic,51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,In stock,1,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...


In [31]:
df_pagina1.to_csv("libros_toscrape_pag1.csv", index=False, encoding="utf-8-sig")
df:pagina1.toexcel("libros_toscrape_pag1.xlsx", index=False)

print("Archivos guardados?")


Archivos guardados?
